# GPS-Less Localization — Model Testing


**Required files (from training notebook):**
- `final_model.pkl`
- `final_scaler.pkl`
- `feature_columns.csv`
- `model_config.json`

## 0. Configuration

In [ ]:
# ── MODEL ARTIFACTS (same for both modes) ─────────────────────────────────────
MODEL_DIR        = "."                        # Folder where training saved artifacts
MODEL_PATH       = f"{MODEL_DIR}/final_model.pkl"
SCALER_PATH      = f"{MODEL_DIR}/final_scaler.pkl"
FEAT_COLS_PATH   = f"{MODEL_DIR}/feature_columns.csv"
CONFIG_PATH      = f"{MODEL_DIR}/model_config.json"

# ── MODE 1: CSV testing ────────────────────────────────────────────────────────
TEST_CSV         = "./DataSet/test_radio_data.csv"      # Raw packets CSV with ground truth

# ── MODE 2: Live serial ────────────────────────────────────────────────────────
SERIAL_PORT      = 'COM5'
BAUD_RATE        = 115200
COLLECTION_TIME  = 15                         # seconds to collect packets

# ── RADIO CONSTANTS ───────────────────────────────────────────────────────────
VALID_ANCHORS        = ['A1', 'A2', 'A3', 'A4', 'A5']
RSSI_FLOOR           = -120.0
SNR_FLOOR            = -20.0
EXPECTED_LORA_COUNT  = 30

## 1. Imports & Shared Utilities

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib, json, warnings
from itertools import combinations

warnings.filterwarnings('ignore')


# ── Haversine distance ─────────────────────────────────────────────────────────
def haversine_m(la1, lo1, la2, lo2):
    R = 6_371_000.0
    a = (np.sin(np.radians((la2 - la1) / 2)) ** 2 +
         np.cos(np.radians(la1)) * np.cos(np.radians(la2)) *
         np.sin(np.radians((lo2 - lo1) / 2)) ** 2)
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


# ── IQR outlier filter ─────────────────────────────────────────────────────────
def iqr_filter(series, k=1.5):
    if len(series) < 4:
        return series
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return series[(series >= q1 - k * iqr) & (series <= q3 + k * iqr)]


# ── Feature extraction: raw packets → feature vector ──────────────────────────
def extract_features(raw_df, feat_cols,
                     rssi_floor=RSSI_FLOOR, snr_floor=SNR_FLOOR,
                     expected_lora=EXPECTED_LORA_COUNT,
                     valid_anchors=None):
    """
    Converts a raw packet DataFrame (one location) into a feature vector
    matching the exact column order expected by the model.

    Parameters
    ----------
    raw_df : DataFrame with columns [anchor, protocol, rssi, snr, seq]
    feat_cols : list of feature names (from feature_columns.csv)

    Returns
    -------
    np.ndarray of shape (1, n_features)
    """
    if valid_anchors is None:
        valid_anchors = VALID_ANCHORS

    MISSING_FILL = {
        'BLE':  ['mean', 'std', 'min', 'max', 'count'],
        'LORA': ['mean', 'std', 'min', 'max', 'count', 'rate', 'snr_mean', 'snr_std'],
        'WIFI': ['mean', 'std', 'min', 'max', 'count'],
    }

    # Clean: valid anchors, RSSI range
    df = raw_df[raw_df['anchor'].isin(valid_anchors)].copy()
    df = df[(df['rssi'] >= -130) & (df['rssi'] <= -30)]

    row = {}

    # Base per-anchor-per-protocol stats
    for anchor in valid_anchors:
        adf = df[df['anchor'] == anchor]
        for proto in ['BLE', 'LORA', 'WIFI']:
            pdf  = adf[adf['protocol'] == proto]
            pfx  = f'{anchor}_{proto}'

            if len(pdf) == 0:
                for stat in MISSING_FILL[proto]:
                    if stat in ('count', 'rate', 'std', 'snr_std'):
                        row[f'{pfx}_{stat}'] = 0.0
                    elif stat == 'snr_mean':
                        row[f'{pfx}_{stat}'] = snr_floor
                    else:
                        row[f'{pfx}_{stat}'] = rssi_floor
            else:
                r = iqr_filter(pdf['rssi'])
                n = len(r)
                row[f'{pfx}_mean']  = r.mean()          if n > 0 else rssi_floor
                row[f'{pfx}_std']   = r.std()           if n > 1 else 0.0
                row[f'{pfx}_min']   = r.min()           if n > 0 else rssi_floor
                row[f'{pfx}_max']   = r.max()           if n > 0 else rssi_floor
                row[f'{pfx}_count'] = n

                if proto == 'LORA':
                    row[f'{pfx}_rate'] = n / expected_lora
                    s = iqr_filter(pdf['snr'])
                    row[f'{pfx}_snr_mean'] = s.mean() if len(s) > 0 else snr_floor
                    row[f'{pfx}_snr_std']  = s.std()  if len(s) > 1 else 0.0

    # DIFF features (LoRa)
    for a1, a2 in combinations(valid_anchors, 2):
        key = f'DIFF_LORA_{a1}_{a2}'
        if key in feat_cols:
            row[key] = row.get(f'{a1}_LORA_mean', rssi_floor) - row.get(f'{a2}_LORA_mean', rssi_floor)

    # DIFF features (BLE)
    for a1, a2 in [('A1', 'A3'), ('A1', 'A5'), ('A3', 'A5')]:
        key = f'DIFF_BLE_{a1}_{a2}'
        if key in feat_cols:
            row[key] = row.get(f'{a1}_BLE_mean', rssi_floor) - row.get(f'{a2}_BLE_mean', rssi_floor)

    # Return in correct column order
    vec = np.array([row.get(c, rssi_floor) for c in feat_cols], dtype=float)
    return vec.reshape(1, -1)


# ── Load model artifacts ───────────────────────────────────────────────────────
model     = joblib.load(MODEL_PATH)
scaler    = joblib.load(SCALER_PATH)
feat_cols = pd.read_csv(FEAT_COLS_PATH)['feature'].tolist()

with open(CONFIG_PATH) as f:
    config = json.load(f)

print("Model loaded successfully.")
print(f"  KNN k           : {config['knn_k']}")
print(f"  Training points : {config['n_training_points']}")
print(f"  CV MAE          : {config['cv_mae_m']} m")
print(f"  CV Median       : {config['cv_median_m']} m")
print(f"  CV P90          : {config['cv_p90_m']} m")
print(f"  Features        : {len(feat_cols)}")

---
**Run cells in this section to connect to the ESP32, collect packets in real time, and predict location.**

**Steps:**
1. Make sure `SERIAL_PORT` in the config cell is set to your port (default: COM5)
2. Run the cell below — it will:
   - Request GPS coordinates from the ESP32
   - Collect radio packets for `COLLECTION_TIME` seconds
   - Predict and display the location

In [ ]:
import serial
import time

def get_gps_location(ser, timeout=5):
    """
    Sends 'G' command to ESP32 and waits for GPS_OK:<lat,lon> response.
    Silently discards background radio packets while waiting.
    Returns coordinate string or None.
    """
    print("Requesting GPS coordinates from ESP32...", end="", flush=True)
    ser.reset_input_buffer()
    ser.write(b'G')

    deadline = time.time() + timeout
    while time.time() < deadline:
        if ser.in_waiting:
            line = ser.readline().decode('utf-8', errors='ignore').strip()
            if not line:
                continue
            if line.startswith("GPS_OK:"):
                coords = line.split(":", 1)[1].strip()
                print(f" Got: {coords}")
                return coords
            elif line == "GPS_WAITING":
                print("\n  ⚠ GPS has no satellite lock. Move outside and retry.")
                return None
            # else: radio packet — ignore silently

    print("\n  ✗ Timeout — no GPS response. Check wiring and port.")
    return None


def collect_live_packets(ser, duration_s):
    """
    Collects valid radio packets from the serial port for duration_s seconds.
    Returns a DataFrame of packets.
    """
    VALID_PROTOCOLS = {"BLE", "LORA", "WIFI"}

    def is_valid(parts):
        if len(parts) != 6:
            return False
        if parts[2] not in VALID_PROTOCOLS:
            return False
        try:
            int(parts[0]);   float(parts[3])
            float(parts[4]); int(parts[5])
        except ValueError:
            return False
        return True

    packets = []
    ser.reset_input_buffer()
    deadline = time.time() + duration_s

    print(f"\nCollecting packets for {duration_s}s — stay still...")
    while time.time() < deadline:
        remaining = deadline - time.time()
        if ser.in_waiting:
            raw = ser.readline().decode('utf-8', errors='ignore').strip()
            if not raw:
                continue
            parts = raw.split(',')
            if is_valid(parts):
                packets.append({
                    'anchor':   parts[1],
                    'protocol': parts[2],
                    'rssi':     int(parts[3]),
                    'snr':      float(parts[4]),
                    'seq':      int(parts[5]),
                })
                print(f"  [{len(packets):3d}] {raw}  ({remaining:.0f}s left)",
                      end='\r')

    print(f"\nCollection done — {len(packets)} valid packets received.")
    return pd.DataFrame(packets) if packets else pd.DataFrame(
        columns=['anchor', 'protocol', 'rssi', 'snr', 'seq'])


print("Mode 2 functions defined. Run the cell below to start live collection.")

In [ ]:
# ── LIVE COLLECTION + PREDICTION ───────────────────────────────────────────────
# This is the main Mode 2 cell. Run it each time you want a prediction.

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    print(f"Connected to {SERIAL_PORT} at {BAUD_RATE} baud")
    time.sleep(2)   # Let ESP32 boot if just connected
    ser.reset_input_buffer()
except Exception as e:
    print(f"\n✗ Could not open {SERIAL_PORT}: {e}")
    raise

try:
    # ── Step 1: Get GPS ground truth ───────────────────────────────────────────
    gps_str = get_gps_location(ser)

    if gps_str is None:
        print("Aborting — no GPS fix.")
    else:
        gt_lat, gt_lon = map(float, gps_str.split(','))

        # ── Step 2: Collect radio packets ──────────────────────────────────────
        print(f"\nGround truth: lat={gt_lat:.6f}, lon={gt_lon:.6f}")
        packets_df = collect_live_packets(ser, COLLECTION_TIME)

        if packets_df.empty:
            print("\n✗ No valid packets received — cannot predict.")
        else:
            # ── Step 3: Extract features & predict ────────────────────────────
            X_vec    = extract_features(packets_df, feat_cols)
            X_scaled = scaler.transform(X_vec)
            pred     = model.predict(X_scaled)[0]
            pred_lat, pred_lon = pred[0], pred[1]
            error_m  = haversine_m(gt_lat, gt_lon, pred_lat, pred_lon)

            # ── Step 4: Display results ────────────────────────────────────────
            print()
            print("=" * 50)
            print("  PREDICTION RESULT")
            print("=" * 50)
            print(f"  Ground truth : lat={gt_lat:.6f},  lon={gt_lon:.6f}")
            print(f"  Predicted    : lat={pred_lat:.6f},  lon={pred_lon:.6f}")
            print(f"  Error        : {error_m:.2f} m")
            print(f"  Packets used : {len(packets_df)} total")

            for proto in ['LORA', 'BLE', 'WIFI']:
                n = (packets_df['protocol'] == proto).sum()
                print(f"    {proto:<5}: {n}")
            print("=" * 50)

            # ── Step 5: Plot ───────────────────────────────────────────────────
            fig, ax = plt.subplots(figsize=(6, 5))
            ax.scatter([gt_lon],   [gt_lat],   c='green',  s=150, zorder=5,
                       label=f'Ground truth\n({gt_lat:.6f}, {gt_lon:.6f})', marker='o')
            ax.scatter([pred_lon], [pred_lat], c='red',    s=150, zorder=5,
                       label=f'Predicted\n({pred_lat:.6f}, {pred_lon:.6f})', marker='^')
            ax.annotate("", xy=(pred_lon, pred_lat), xytext=(gt_lon, gt_lat),
                        arrowprops=dict(arrowstyle='->', color='navy', lw=2))
            ax.set_title(f'Live Prediction — Error: {error_m:.2f} m')
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude')
            ax.legend(fontsize=9)
            ax.grid(alpha=0.3)
            plt.tight_layout()
            plt.savefig('mode2_live_result.png', dpi=150, bbox_inches='tight')
            plt.show()
            print("Saved: mode2_live_result.png")

finally:
    ser.close()
    print("Serial port closed.")

In [ ]:
# ── OPTIONAL: Multi-point live session ─────────────────────────────────────────
# Same as above but loops until you type 'exit'
# Keeps a running record of all predictions in this session.

session_results = []

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    print(f"Connected to {SERIAL_PORT}. Type 'exit' at any prompt to stop.")
    time.sleep(2)
    ser.reset_input_buffer()
except Exception as e:
    print(f"✗ Could not open {SERIAL_PORT}: {e}")
    raise

try:
    while True:
        cmd = input("\nPress Enter to collect at this location (or type 'exit'): ")
        if cmd.strip().lower() == 'exit':
            break

        gps_str = get_gps_location(ser)
        if gps_str is None:
            print("No GPS fix — skipping.")
            continue

        gt_lat, gt_lon = map(float, gps_str.split(','))
        packets_df = collect_live_packets(ser, COLLECTION_TIME)

        if packets_df.empty:
            print("No valid packets — skipping.")
            continue

        X_vec    = extract_features(packets_df, feat_cols)
        X_scaled = scaler.transform(X_vec)
        pred     = model.predict(X_scaled)[0]
        pred_lat, pred_lon = pred[0], pred[1]
        error_m  = haversine_m(gt_lat, gt_lon, pred_lat, pred_lon)

        session_results.append({
            'gt_lat': gt_lat, 'gt_lon': gt_lon,
            'pred_lat': pred_lat, 'pred_lon': pred_lon,
            'error_m': error_m, 'n_packets': len(packets_df),
        })

        print(f"  Ground truth : ({gt_lat:.6f}, {gt_lon:.6f})")
        print(f"  Predicted    : ({pred_lat:.6f}, {pred_lon:.6f})")
        print(f"  Error        : {error_m:.2f} m  |  Running MAE: "
              f"{np.mean([r['error_m'] for r in session_results]):.2f} m")

finally:
    ser.close()
    print("Serial port closed.")

if session_results:
    sess_df = pd.DataFrame(session_results)
    print(f"\n{'=' * 45}")
    print(f"  SESSION SUMMARY  ({len(sess_df)} points)")
    print(f"{'=' * 45}")
    print(f"  MAE    : {sess_df['error_m'].mean():.2f} m")
    print(f"  Median : {sess_df['error_m'].median():.2f} m")
    print(f"  Max    : {sess_df['error_m'].max():.2f} m")
    print(f"{'=' * 45}")